# 3.1 - Brand classification. 
This notebook follows after applying the `../model appliers/2 -apply_angletag.ipynb` notebook.

In a third phase the notebooks will focus on predicting the correct 'brand' of a car. These brands are tags that were present at the scraping phase and no inference or propagation is required. 

There are multiple notebooks in this phase - each referenced to by a second digit in the format of `x.y - Brand classification` where `x` coincides with the phase and will always be three and `y` coincides with a unique notebook being used within this phase. References in this phase will always have this `x.y` notation. Within a notebook sections are labeled with `x.y.z. <title>`; the `x` and `y` values are constant in the notebook; the `z` element is the section within the notebook. 

Three different approaches will be explored in phase 3; each approach has it's own distinct model being trained. Given the long time it took to train 20.000 images in phase 2 (up to 45 minutes) for a relatively small set - this phase will explore techniques that come to results more quickly than making a model based on the KERAS API from scratch. 

 - notebook `3.2. - TINY Brand classification: transfer learning (angled).ipynb` will use the front angle and a small subset of 5 brands to test if cropping an image improves accuracy of the model. The outcome of this will be applied to notebooks `3.3`, `3.4` and `3.5`

 - notebook `3.3. - Brand classification: transfer learning (angled).ipynb` will use the angle tags generated in phase two and output one model per angle. The full prediction pipeline is then: 
    - Apply YOLO box (if applicable `3.2`)
    - assess Usability - open for discussion
    - predict angle
    - use model of trained angle to predict brand. 

- notebook `3.4. - Brand classification: transfer learning (unangled).ipynb` will not use the angle tags and will be used to train the full dataset; the goal is to predict one brand per image. The full prediction pipeline up until this point is then: 
    - Apply YOLO box (if applicable `3.2`)
    - assess Usability - open for discussion
    - use model of trained angle to predict brand. 

- notebook `3.5. - Brand classification: reinforcement learning.ipynb` will use reinforcement learning to try and predict the image - this is not a standard approach to image recognition problems, but it was an experiment I wanted to try based on the interviews being made in newspapers following the DeepSeek launch: 
    > Nieuwe trainingsmethode van DeepSeek “Een andere fundamentele verandering die DeepSeek heeft geïntroduceerd is de manier waarop AI-modellen worden getraind. Waar de meeste Large Language Models vertrouwen op enorme hoeveelheden gelabelde data en Supervised Fine-Tuning, heeft DeepSeek-R1 laten zien dat dit sterke redeneervermogen ook kan worden bereikt met een aanpak die puur gebaseerd is op Reinforcement Learning (RL).
    
    Source: [manners.nl](https://www.manners.nl/deepseek-ai-muur-budget-big-tech-open-source/)

    > Why it matters: Reinforcement learning has surprising utility in training large language models to reason. As researchers press models into service in more complex tasks — math, coding, animated graphics, and beyond — reinforcement learning is emerging as an important path to progress.

    Source: [deeplearning.ai](https://www.deeplearning.ai/the-batch/how-deepseek-r1-and-kimi-k1-5-use-reinforcement-learning-to-improve-reasoning/)


- notebook `3.6. - Brand scoring.ipynb` will use the output of notebooks `3.3`, `3.4` and `3.5` to asses the best approach and pitch the performance of an establish method (transfer learning) against a method that's not typically recommended for image classification tasks: ((reinforcement learning)). In the final scoring notbeook we'll also aim to answer the question whether or not it's worth to train an angle classifier and use models trained on less datapoints, but more homogenous points versus a model that was trained on a large batch of data that's more heterogenous.


The difficulty will be to use the same data and do this efficiently in three different notebooks. (i.e. do a single pass of augmentation versus 3 separate augmentation phases). Have the same splits being applied everywhere etc... 

To guarantee this; the current notebook will handle all augmentation and splitting needs for the model. The output of this notebook is a CSV dump of all files with augmented data being generated and included in notebook `3.1`. 

Since we plan on using `resnet50` in notebooks `3.3` and `3.4` that means we have to use a fixed training shape of `224 * 224`, for the sake of uniformity this shape requirement is used across ALL notebooks in phase three - including the `3.5` notebook where RL is used.

In [1]:
import pandas as pd
import os
import sys
#import re
import random
import uuid
from tqdm import tqdm


sys.path.append('../../utils')
from  configloader import Configloader
from database import Database

import cnn_helpers
import file_io
import repository

2025-04-06 17:13:27.864439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
config = Configloader()

db = Database(config)
db.connect()

Connection established


In [3]:
basedir = config.get('settings', 'image_directory')
augmdir = config.get('dir_augmentations', 'subfolder')
augmcsv = config.get('dir_augmentations', 'csv_dir')


In [4]:
augment_base = os.path.join(basedir, augmdir)
augment_csv_dump = os.path.join(basedir, augmcsv)


## 3.1.1. Wipe slate: 
Clean out the augmentation folder and wipe the augmented data.

In [4]:
file_io.wipe_folder(augment_base)
file_io.wipe_folder(augment_csv_dump)

## 3.1.2 Making distribution decisions
we need some way of making our classes balanced, we know from the EDA phase that there are heavy imbalances between certain brands of the cars in this dataset. For instance BMW is a very popuplar brand whereas Lotus or Alpine are lesser known brands and thus less prevalent in the dataset.

We'll adress this by using augmentation again, when training the angle model it becam apparant that the model performance could shift about 0.7% depending on the random choices made at various points in the training process (which images to train/test split, which images to augmentate, what values to use in the augmentation process). To prevent this kind of drift between the notebooks `3.3`, `3.4` and `3.5`; we'll use the current notebook `3.1` to do all splitting operations, and all augmentation operations. 

This has the added benefit of being more efficient: The augmentation happens ONCE and is then re-used across different notebooks.

In [5]:
#what we learned from phase 1: 
BINMODELS_PASS = 2
BINMODELS_MINSCORE = 0.9

#how many samples: per brand per angle
BRAND_ANGLE_COMBO_SAMPLES = 10000   #augment where needed; undersample where needed.

#SPLIT RATIOS: in percentual values expressed sum should be 100 
#Train - Test - Validation: 
VALIDATIONSAMPLE = 10
TESTSAMPLE = 20
TRAINSAMPLE = 70

#WHERE TO STORE CSV DUMP DATA: 



## 3.1.3. Getting data from SQL
Take the data out of SQL and prepare it: address the bounding boxes and image paths.

In [6]:
data = pd.DataFrame(
    db.execute_query(repository.get_images(BINMODELS_MINSCORE, BINMODELS_PASS))
)
data = file_io.make_absolute_path(data, basedir, 'image_path', 'abs_path')
data = cnn_helpers.cast_bbox_values(data)

In [7]:
data.sample(5)

,image_id,model_label,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand,abs_path
2993949,11588682,right,1.000000,98,232,675,462,11588682,0.998207,0.999982,0.981016,0.991167,hyundai,/home/frederic/Documents/automotive_image_data...
797619,3107517,frontright,1.000000,92,256,611,547,3107517,0.979693,0.988293,0.623558,0.543022,audi,/home/frederic/Documents/automotive_image_data...
1035611,3945821,rearright,1.000000,71,152,676,481,3945821,0.996994,0.999965,0.986238,0.986905,volkswagen,/home/frederic/Documents/automotive_image_data...
3165068,12378522,rearleft,0.999997,27,146,731,471,12378522,0.988950,0.938504,0.692716,0.945982,volkswagen,/home/frederic/Documents/automotive_image_data...
950716,3647922,front,1.000000,173,99,577,423,3647922,0.985720,0.901154,0.465477,0.589085,audi,/home/frederic/Documents/automotive_image_data...


## 3.1.4 Splitting data in train-test-split
Here we'll be splitting the `data` dataframe as defined by the split ratio constants. Split happens using a stratified method that uses the `brand` and `model_label` columns. After stratification the data validation data gets dumped out and is not used until after the training phase. It'll also be used in the `3.5` notebook where the performance of the three models are measured on the same validation set!

Test data and validation data will NOT be augmented.
Train data will be augmented using the same augmentation techniques used before. With an added flipping method over the y-axis for sideviews. This is a quick win to address the extreme class imbalance. NOTE THAT this will actually be most beneficial to the  `3.2` notebook where a model is trained per angle; in other notebooks I expect less gains by this method. 



In [8]:
stratcols = ['brand', 'model_label']
trainset, testset, valset = cnn_helpers.train_test_val_splitter(
                                data, 
                                TRAINSAMPLE,
                                TESTSAMPLE,
                                VALIDATIONSAMPLE, 
                                stratcols
                            )

## 3.1.5 Apply data-augmentation/undersampling to trainset
There's a global upperlimit `BRAND_ANGLE_COMBO_SAMPLES` on how much rows we want to have per brand/angle combination: let's apply that here to the trainset. 

Sometimes there's plenty of data for a single brand, other times, we'll be needing augmentation: go over the dataframe brand by brand, angle by angle and see if you have enough data: in that case, randomly subsample from that (undersample) and store the subsample. Otherwise, augmentate. 

As far as augmentation goes: there's one kind of augmentation that should be preferred for the left-right issue: these angles are the most underrepresented; an easy fix is to swap them over the y-axis of an image. We'll only do this for images where the prediction is 98% sure or more. 

This method creates synthetic data for the least occuring angles and is contained to the trainingdata only. For other angles we'll not be using this approach.  

In [9]:
#CSV to dump: 
sampled_rows = []

def to_sample(df, into): 
    for idx, row in df.iterrows():
        into.append(row)

#step 1: check if you have the minimum amount of rows for a given brand/angle combo
for combo, df in trainset.groupby(stratcols):
    brand = combo[0]
    angle = combo[1]
    df = cnn_helpers.shuffle_df(df)
    sideviews = ['left', 'right']
    if angle in sideviews:
        opposite = 'right' if angle == 'left' else 'left'
    #easy: more data than required: shuffle df and take the first x images.
    if len(df) >= BRAND_ANGLE_COMBO_SAMPLES: 
        pass
        to_sample(df[0:BRAND_ANGLE_COMBO_SAMPLES], sampled_rows)
    #darn: there's not enough.
    else:
        targetpath = os.path.join(augment_base, brand)
        add_to_sampled = []
        #how many samples are lacking:
        # start by adding whatever is not-agumented and available:
        to_sample(df, add_to_sampled)
        # if you have natural side views: implement mirroring, all other views, skip mirroring phase.
        remaining = BRAND_ANGLE_COMBO_SAMPLES - len(add_to_sampled)
        if angle in sideviews:
            opposite_view_df = trainset.query('model_score >= 0.98 and model_label == @opposite and brand == @brand')
            opposite_view_df = cnn_helpers.shuffle_df(opposite_view_df)
            #when mirroring: do not exceed the BAC_SAMPLES constant
            if len(opposite_view_df) >= remaining:
                opposite_view_df = opposite_view_df[0:remaining]
            for idx, row in opposite_view_df.iterrows(): 
                sourcepath = row['abs_path']
                uuid_v4 = str(uuid.uuid4())
                new_name = uuid_v4 + '.' + sourcepath.split('.')[-1]
                new_angle = cnn_helpers.augm_mirror_image(sourcepath, targetpath, new_name, opposite)
                row_modded = row.copy()
                row_modded['abs_path'] = os.path.join(targetpath, new_name)
                row_modded['model_label'] = new_angle
                add_to_sampled.append(row_modded)
        while len(add_to_sampled) < BRAND_ANGLE_COMBO_SAMPLES:
            # at the very least augment should be made a utility - be carefull, it takes a df as arg, not a row!!
            # the df in the arg has a lenght of 1. (sample.)                
            sampled_row = df.sample(1)
            augmentation_results = cnn_helpers.augment(sampled_row, targetpath)
            add_to_sampled.append(augmentation_results)
        sampled_rows.extend(add_to_sampled[0:BRAND_ANGLE_COMBO_SAMPLES])



In [10]:
augemnted_df = pd.DataFrame(sampled_rows)

In [11]:
#Test if every combo meets the exact number of training instances. 
combo_counts = augemnted_df.groupby(stratcols).size()
good_augmentation = (combo_counts == BRAND_ANGLE_COMBO_SAMPLES).all()
if good_augmentation:
    print(f'All combinations of columns {stratcols} have {BRAND_ANGLE_COMBO_SAMPLES} rows.')
else:
    print('Not all combination of rows meets the exact number of rows. Proceed with caution')

All combinations of columns ['brand', 'model_label'] have 10000 rows.


In [12]:
file_io.table_to_csv(augemnted_df, augment_csv_dump, 'traindata_brandphase.csv')
file_io.table_to_csv(testset, augment_csv_dump, 'testdata_brandphase.csv')
file_io.table_to_csv(valset, augment_csv_dump, 'validationdata_brandphase.csv')


In [13]:
print(len(augemnted_df))

2400000


In [14]:
combo_counts

brand       model_label
alfa-romeo  front          10000
            frontleft      10000
            frontright     10000
            left           10000
            rear           10000
                           ...  
volvo       left           10000
            rear           10000
            rearleft       10000
            rearright      10000
            right          10000
Length: 240, dtype: int64

There's an important detail to remember. The data has balanced angles. In case notebook `3.3` and `3.4` produce a similar precision, we'll want to test if a model trained on unbalanced angles is as performant. We'll not prepare the data for this now. Should we need it, we'll  be doing a selection of our data of 800000 randomly chosen images per brand, with augmentation where it is needed. since we want to compare the performance of one large model without angle information versus 8 smaller models with balanced angles we'll not look at the actual distribution of the angle in this data. Since  I'm expecting this to take a long computation time. I'll only consider this step if notebook `3.6` does not show a clear performance difference between step `3.3` and `3.4`!!

TLDR: the output of nthsis notebook gives you balanced angles. If you truly want to compare model performance without this feature, the trainingdata for notebook `3.4` shouldn't have this feature . If we already see a perforrmance drop as it is now, we'll not be running `3.4`  with unbalanced angles!!